# Experimento — `<modelo>` (`<seu nome>`)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gpeuzinCosta/kws-borda/blob/main/notebooks/experimentos/template_modelo.ipynb)

**Rascunho de desenvolvimento.** Copie este arquivo para `notebooks/experimentos/<modelo>_<nome>.ipynb` e use
para implementar e ajustar o seu modelo. Os **resultados finais** não ficam aqui: eles vão para o notebook
principal `kws_borda_relatorio.ipynb`, que roda todos os modelos com as mesmas regras.

Roteiro:
1. Implemente o modelo em `modelos/<modelo>.py` (a docstring traz a arquitetura-alvo) e mude `IMPLEMENTADO = True`.
2. Teste o contrato e o custo aqui, com **dados falsos** (rápido, sem baixar nada).
3. Treine no GSC de verdade e ajuste os hiperparâmetros no `configs/<modelo>.yaml`.
4. Faça o commit do código e do YAML e abra um PR.

> Dica: se você editar o `.py` do modelo no Colab (painel de arquivos, em `/content/kws-borda/modelos/...`), rode
> `importlib.reload` ou reinicie o kernel para pegar a mudança.

In [ ]:
#@title Setup — rode esta célula primeiro
#Funciona no Colab (clona o repo e usa o Drive) e localmente/VS Code (usa a pasta do repositório).
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/gpeuzinCosta/kws-borda.git"
NO_COLAB = "google.colab" in sys.modules

if NO_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO = Path("/content/kws-borda")
    if REPO.exists():
        subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)     #pega a versão mais nova
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)

    #Drive: cache de MFCC, runs e resultados PERSISTEM entre sessões
    #Disco local (/content/data): áudios extraídos, leitura rápida (o Drive é lento para 100 mil arquivos)
    os.environ["KWS_DRIVE"] = "/content/drive/MyDrive/kws-borda"
    os.environ["KWS_DATA"] = "/content/data"
else:
    #Local: sobe a partir da pasta atual até achar a raiz do repositório
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())

#A raiz do repo no sys.path deixa importar as pastas: configs, dados, modelos, treino, avaliacao
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import torch
from configs import pastas_projeto, escolher_device

PASTAS = pastas_projeto()
DOWNLOADS = Path(os.environ.get("KWS_DRIVE", REPO)) / "downloads"     #.tar.gz/.zip baixados UMA vez
device = escolher_device()
commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()

print(f"Colab: {NO_COLAB} | device: {device} | commit do código: {commit}")
print(f"torch {torch.__version__} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '—'}")
for nome, p in PASTAS.items():
    print(f"  {nome:10s} -> {p}")

## Hiperparâmetros

O `configs/base.yaml` + `configs/<modelo>.yaml` viram um único dicionário. Para testar um valor, sobrescreva
aqui. Quando achar o melhor, passe-o para o YAML.

In [ ]:
from configs import PRESETS_MFCC, carregar_config

NOME_MODELO = "mlp"         #troque pelo seu: cnn1d, resnet, dscnn, atencao
DADOS_FALSOS = True         #True: MFCC aleatório (testa o pipeline em segundos)
TAREFA = "gsc35"            #"gsc35" ou "gsc12" (10 comandos + unknown + silence)
MFCC = "padrao"             #"padrao" (40x101) ou "hello_edge" (10x49)

hyperparameters = carregar_config(NOME_MODELO, tarefa=TAREFA, **PRESETS_MFCC[MFCC],
    normalizacao=None,      #None mantém o YAML ("global"); ou "por_amostra"
    epocas=2,               #sobrescritas para experimentar; None mantém o valor do YAML
    learning_rate=None,
)
hyperparameters

## Contrato e custo do modelo

Entrada `(B, n_mfcc, n_frames)` → saída `(B, n_classes)`. Confira o número de parâmetros e de MACs contra o alvo da
docstring.

In [ ]:
import torch
from modelos import criar_modelo
from avaliacao import custo

model = criar_modelo(NOME_MODELO, hyperparameters)
x = torch.randn(4, hyperparameters["n_mfcc"], hyperparameters["n_frames"])
print("entrada:", tuple(x.shape), "-> saída:", tuple(model.eval()(x).shape))

c = custo(model)
print(f"parâmetros: {c['params']:,} | {c['kb_fp32']:.0f} KB fp32 | {c['kb_int8']:.0f} KB int8 | MACs: {c['macs']:,}")

In [ ]:
#Resumo camada a camada (formato da saída e parâmetros de cada uma)
!pip install torchinfo -q
import torchinfo
torchinfo.summary(model, input_size=(2, hyperparameters["n_mfcc"], hyperparameters["n_frames"]))

## Dados

In [ ]:
import subprocess, sys
from dados import carregar_gsc, criar_dataloaders, dados_falsos

if DADOS_FALSOS:
    dados = dados_falsos(hyperparameters, n=512)
else:
    subprocess.run([sys.executable, str(REPO / "dados" / "baixar.py"), "--dataset", "gsc",
                    "--destino", str(PASTAS["dados"]), "--arquivos", str(DOWNLOADS)], check=True)
    dados = carregar_gsc(hyperparameters, PASTAS["dados"] / "gsc", PASTAS["cache"])

loaders, (MEDIA, DESVIO) = criar_dataloaders(hyperparameters, dados)
xb, yb = next(iter(loaders["treino"]))
print("batch:", tuple(xb.shape), yb[:8].tolist())

## Treino (loop unificado)

In [ ]:
%%time
from treino import treinar, prever
from avaliacao import acuracia, f1_macro
from avaliacao.graficos import curvas_treino

model, hist = treinar(criar_modelo(NOME_MODELO, hyperparameters), loaders, hyperparameters, seed=0,
                      pasta_runs=None if DADOS_FALSOS else PASTAS["runs"], device=device)
curvas_treino(hist, titulo=f"({NOME_MODELO})")

y_pred, y_true = prever(model, loaders["teste"])
print(f"teste | acurácia: {acuracia(y_pred, y_true)*100:.2f}% | F1 macro: {f1_macro(y_pred, y_true):.4f}")

## Latência (MFCC × rede)

In [ ]:
from dados import ExtratorMFCC
from avaliacao import medir_latencia

r = medir_latencia(model, ExtratorMFCC.de_hparams(hyperparameters), threads=1, repeticoes=100)
print(f"MFCC: {r['mfcc_ms']:.3f} ms | rede: {r['rede_ms']:.3f} ms | MFCC/rede = {r['mfcc_sobre_rede']:.2f}x")

## Anotações do experimento

| Data | Mudança | Params | Acurácia val | Observação |
|---|---|---|---|---|
| | | | | |